# CS-323 Artificial Intelligence: Complex Engineering Problem (CEP)
## Term Project: Intelligent Vacuum-Cleaning Robot

**Group Members:**
1. **[Member 1 Name]** — `[CS-24XXX]`
2. **[Member 2 Name]** — `[CS-24XXX]`
3. **[Member 3 Name]** — `[CS-24XXX]`
4. **[Member 4 Name]** — `[CS-24XXX]`

---
### Project Overview
This notebook implements an **Intelligent Vacuum-Cleaning Robot** operating in a 2D grid environment. It integrates two classical Artificial Intelligence paradigms:
1. **Constraint Satisfaction Problem (CSP):** Determines a valid cleaning order across available time slots while satisfying precedence, latest-slot, and slot-exclusivity constraints using **Backtracking Search**, **MRV + Degree heuristics**, **LCV ordering**, and **Forward Checking**.
2. **Graph-Based Search:** Computes collision-free 4-directional paths between the charging station and dirty locations using **Breadth-First Search (BFS)** (uninformed) and **A\* Search** with Manhattan distance heuristic (informed).

## 0. Input Files Setup & Verification
The required `.txt` files (`Environment.txt`, `Schedule.txt`, `Environment_Sample.txt`, `Schedule_Sample.txt`, `Environment_Error.txt`, and `Schedule_Error.txt`) are loaded dynamically at runtime. The cell below verifies/creates these input files in the working directory so the notebook can be executed end-to-end without hardcoded parameters.

In [1]:
import collections
import heapq
import copy
import re
import os

# Create the 6 required input files if not already present in the working directory
input_files = {
    "Environment.txt": """10,10
12
(1,2) (1,3) (2,3) (3,3) (4,3) (4,6) (5,6) (6,6) (7,2) (7,3) (8,5) (8,6)
5
(0,9) (3,8) (8,8) (9,1) (5,2)
(0,0)""",
    "Schedule.txt": """5
4
D1,D3
D2,D4
D1,D5
D5,D4
3
D2,T3
D1,T2
D3,T4""",
    "Environment_Sample.txt": """8,8
6
(1,3) (2,3) (3,3) (4,3) (5,3) (6,3)
4
(0,7) (3,7) (7,7) (7,0)
(0,0)""",
    "Schedule_Sample.txt": """4
3
D1,D3
D2,D4
D1,D4
2
D3,T3
D2,T3""",
    "Environment_Error.txt": """8,8
8
(1,3) (2,3) (3,3) (4,3) (5,3) (6,3) (0,6) (1,7)
4
(0,7) (3,7) (7,7) (7,0)
(0,0)""",
    "Schedule_Error.txt": """5
3
D1,D2
D2,D3
D3,D1
2
D1,T3
D2,T4"""
}

for filename, content in input_files.items():
    if not os.path.exists(filename):
        with open(filename, "w") as f:
            f.write(content.strip() + "\n")

print("Input files verified in working directory:", [f for f in input_files if os.path.exists(f)])

Input files verified in working directory: ['Environment.txt', 'Schedule.txt', 'Environment_Sample.txt', 'Schedule_Sample.txt', 'Environment_Error.txt', 'Schedule_Error.txt']


## 1. Environment & Schedule Input Parsers
* `parse_environment(file_path)`: Reads grid dimensions (`rows,cols`), obstacle coordinates, dirty location coordinates (assigned labels `D1, D2, ...` in order of appearance), and the charging station coordinate.
* `parse_schedule(file_path, num_dirty)`: Reads available time slots (`T1, T2, ...`), precedence pairs `(D_before, D_after)`, and latest-slot rules `(D_task, T_slot)`. Validates that the number of time slots is at least the number of dirty locations.

In [2]:
def parse_environment(file_path):
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    rows, cols = map(int, lines[0].split(','))
    num_obstacles = int(lines[1])

    # Extract tuples using regex: (r,c)
    obstacle_coords = re.findall(r'\((\d+),(\d+)\)', lines[2])
    obstacles = set((int(r), int(c)) for r, c in obstacle_coords)

    num_dirty = int(lines[3])
    dirty_coords = re.findall(r'\((\d+),(\d+)\)', lines[4])
    dirty_locations = [(int(r), int(c)) for r, c in dirty_coords]

    charging_coord = re.findall(r'\((\d+),(\d+)\)', lines[5])[0]
    charging_station = (int(charging_coord[0]), int(charging_coord[1]))

    grid = {'rows': rows, 'cols': cols, 'obstacles': obstacles}
    return grid, charging_station, dirty_locations


def parse_schedule(file_path, num_dirty):
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    num_slots = int(lines[0])
    if num_slots < num_dirty:
        raise ValueError(f"Number of time slots ({num_slots}) must be >= number of dirty locations ({num_dirty})")

    num_prec = int(lines[1])
    precedence = []
    current_line = 2
    for _ in range(num_prec):
        parts = lines[current_line].split(',')
        precedence.append((parts[0].strip(), parts[1].strip()))
        current_line += 1

    num_latest = int(lines[current_line])
    current_line += 1
    latest_slots = []
    for _ in range(num_latest):
        parts = lines[current_line].split(',')
        latest_slots.append((parts[0].strip(), parts[1].strip()))
        current_line += 1

    constraints = {
        'precedence': precedence,
        'latest_slots': latest_slots,
        'num_slots': num_slots
    }
    return constraints

## 2. Graph-Based Search Module (Robot Navigation)
Implements the required search functions for 4-directional grid navigation (`UP`, `DOWN`, `LEFT`, `RIGHT`) with uniform step cost `1`:
* `get_successors(state, grid)`: Returns valid adjacent `(next_state, action, step_cost)` tuples within grid boundaries and free of obstacles.
* `goal_test(state, goal_state)`: Checks if `state == goal_state`.
* `path_cost(path)`: Computes total movement cost (`len(path) - 1`, or `0` for empty/single-state paths).
* `heuristic(state, goal_state)`: Computes Manhattan distance $h(n) = |r_n - r_{goal}| + |c_n - c_{goal}|$, which is admissible and consistent for 4-connected grids.
* `search(grid, start, goal, algorithm="A*")`: Supports **BFS** (uninformed) and **A\*** (informed), returning `(path, cost, nodes_expanded)`.

In [3]:
def get_successors(state, grid):
    r, c = state
    successors = []
    # Actions: UP, DOWN, LEFT, RIGHT (no diagonal movement)
    moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]

    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < grid['rows'] and 0 <= nc < grid['cols']:
            if (nr, nc) not in grid['obstacles']:
                successors.append(((nr, nc), action, 1))
    return successors


def goal_test(state, goal_state):
    return state == goal_state


def path_cost(path):
    if not path or len(path) <= 1:
        return 0
    return len(path) - 1


def heuristic(state, goal_state):
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def search(grid, start, goal, algorithm="A*"):
    if start == goal:
        return [start], 0, 0

    nodes_expanded = 0

    if algorithm == "BFS":
        frontier = collections.deque([(start, [start])])
        explored = {start}

        while frontier:
            state, path = frontier.popleft()
            nodes_expanded += 1

            for next_state, action, cost in get_successors(state, grid):
                if next_state not in explored:
                    new_path = path + [next_state]
                    if goal_test(next_state, goal):
                        return new_path, path_cost(new_path), nodes_expanded
                    explored.add(next_state)
                    frontier.append((next_state, new_path))

    elif algorithm == "A*":
        # Priority Queue stores: (f_score, state, path, g_score)
        start_h = heuristic(start, goal)
        frontier = [(start_h, start, [start], 0)]
        explored = {}  # state -> best_g_score

        while frontier:
            f, state, path, g = heapq.heappop(frontier)

            if state in explored and explored[state] <= g:
                continue

            explored[state] = g
            nodes_expanded += 1

            if goal_test(state, goal):
                return path, path_cost(path), nodes_expanded

            for next_state, action, step_cost in get_successors(state, grid):
                new_g = g + step_cost
                if next_state not in explored or explored[next_state] > new_g:
                    new_f = new_g + heuristic(next_state, goal)
                    heapq.heappush(frontier, (new_f, next_state, path + [next_state], new_g))

    return None, float('inf'), nodes_expanded

## 3. Constraint Satisfaction Problem (CSP) Module (Task Scheduling)
Formulates cleaning task scheduling as a CSP:
* **Variables:** `D1, D2, ..., Dn` corresponding to dirty locations in `Environment.txt`.
* **Domains:** Available time slots `{T1, T2, ..., Tm}` where $m \ge n$.
* **Constraints:**
  1. *Slot Exclusivity (All-Different):* At most one task per time slot.
  2. *Precedence:* `D_before` must be assigned an earlier slot than `D_after`.
  3. *Latest-Slot:* `D_task` must be assigned a slot $\le$ `T_slot`.
* **Required Functions:** `select_unassigned_variable` (MRV + Degree tie-breaker), `order_domain_values` (LCV), `is_consistent`, `forward_check` (with domain restoration on backtrack), and `backtracking_search`.

In [4]:
def select_unassigned_variable(assignment, domains, constraints):
    unassigned = [v for v in domains.keys() if v not in assignment]

    # MRV: Variable with fewest remaining legal values
    mrv_list = []
    min_domain_size = float('inf')
    for v in unassigned:
        size = len(domains[v])
        if size < min_domain_size:
            min_domain_size = size
            mrv_list = [v]
        elif size == min_domain_size:
            mrv_list.append(v)

    if len(mrv_list) == 1:
        return mrv_list[0]

    # Degree Heuristic: Tie-breaker (most constraints with unassigned neighbors)
    max_degree = -1
    best_var = mrv_list[0]

    for v in mrv_list:
        degree = 0
        for d_before, d_after in constraints['precedence']:
            if (d_before == v and d_after not in assignment) or (d_after == v and d_before not in assignment):
                degree += 1
        # Implicit all-different constraint with all other unassigned variables
        degree += len(unassigned) - 1

        if degree > max_degree:
            max_degree = degree
            best_var = v

    return best_var


def order_domain_values(variable, assignment, domains, constraints):
    values = domains[variable]
    if len(values) <= 1:
        return values

    unassigned = [v for v in domains.keys() if v not in assignment and v != variable]

    def count_eliminated(val):
        count = 0
        t_v = int(val[1:])
        for neighbor in unassigned:
            for n_val in domains[neighbor]:
                if n_val == val:
                    count += 1
                    continue
                t_n = int(n_val[1:])
                for d_before, d_after in constraints['precedence']:
                    if variable == d_before and neighbor == d_after:
                        if not (t_v < t_n):
                            count += 1
                    elif variable == d_after and neighbor == d_before:
                        if not (t_n < t_v):
                            count += 1
        return count

    return sorted(values, key=count_eliminated)


def is_consistent(variable, value, assignment, constraints):
    # 1. Slot Exclusivity (All-Different)
    if value in assignment.values():
        return False

    t_val = int(value[1:])

    # 2. Latest-Slot Constraint
    for d_task, t_slot in constraints['latest_slots']:
        if d_task == variable:
            max_t = int(t_slot[1:])
            if t_val > max_t:
                return False

    # 3. Precedence Constraints
    for d_before, d_after in constraints['precedence']:
        if variable == d_before and d_after in assignment:
            t_after = int(assignment[d_after][1:])
            if not (t_val < t_after):
                return False
        elif variable == d_after and d_before in assignment:
            t_before = int(assignment[d_before][1:])
            if not (t_before < t_val):
                return False

    return True


def forward_check(variable, value, assignment, domains, constraints):
    new_domains = copy.deepcopy(domains)
    t_val = int(value[1:])

    for neighbor in domains.keys():
        if neighbor in assignment:
            continue

        remaining_values = []
        for n_val in new_domains[neighbor]:
            t_n = int(n_val[1:])

            # Rule 1: Slot exclusivity
            if n_val == value:
                continue

            # Rule 2: Precedence
            consistent = True
            for d_before, d_after in constraints['precedence']:
                if variable == d_before and neighbor == d_after:
                    if not (t_val < t_n):
                        consistent = False
                        break
                elif variable == d_after and neighbor == d_before:
                    if not (t_n < t_val):
                        consistent = False
                        break

            if consistent:
                remaining_values.append(n_val)

        if not remaining_values:
            return None
        new_domains[neighbor] = remaining_values

    return new_domains


def backtracking_search(variables, domains, constraints):
    def backtrack(assignment, current_domains):
        if len(assignment) == len(variables):
            return assignment

        var = select_unassigned_variable(assignment, current_domains, constraints)
        for value in order_domain_values(var, assignment, current_domains, constraints):
            if is_consistent(var, value, assignment, constraints):
                assignment[var] = value

                # Forward checking (creates copy so backtracking restores domains automatically)
                new_domains = forward_check(var, value, assignment, current_domains, constraints)
                if new_domains is not None:
                    result = backtrack(assignment, new_domains)
                    if result is not None:
                        return result

                del assignment[var]
        return None

    return backtrack({}, domains)

## 4. System Integration Module
`execute_schedule(grid, charging_station, schedule, algorithm="A*")` links the CSP output with the Graph-Based Search solver:
1. Initializes the robot at `charging_station`.
2. Iteratively navigates to each dirty location in the chronological order produced by the CSP solver, marking each location as cleaned upon arrival.
3. Computes a valid path back to `charging_station` once all dirty cells are cleaned.
4. Concatenates path segments without duplicating shared endpoints and reports failure immediately if any leg is unreachable.

In [5]:
def execute_schedule(grid, charging_station, schedule, algorithm="A*"):
    """
    Executes the CSP-generated schedule using graph-based search.
    schedule: list of (task_label, coordinate) ordered by time slot.
    Returns: (complete_path, total_cost, success)
    """
    current_pos = charging_station
    complete_path = [charging_station]
    total_cost = 0
    cleaned_locations = set()
    leg_details = []

    for task_label, target_pos in schedule:
        path, cost, nodes = search(grid, current_pos, target_pos, algorithm)
        if path is None:
            execute_schedule.last_legs = leg_details
            return complete_path, float('inf'), False

        start_label = leg_details[-1]['to'] if leg_details else 'Charging Station'
        leg_details.append({
            'from': start_label,
            'to': f"{task_label} {target_pos}",
            'cost': cost,
            'nodes': nodes,
            'path': path
        })

        # Append path excluding duplicate starting cell
        complete_path.extend(path[1:])
        total_cost += cost
        current_pos = target_pos
        cleaned_locations.add(target_pos)

    # Return to charging station
    path, cost, nodes = search(grid, current_pos, charging_station, algorithm)
    if path is None:
        execute_schedule.last_legs = leg_details
        return complete_path, float('inf'), False

    leg_details.append({
        'from': leg_details[-1]['to'] if leg_details else 'Charging Station',
        'to': f"Charging Station {charging_station}",
        'cost': cost,
        'nodes': nodes,
        'path': path
    })

    complete_path.extend(path[1:])
    total_cost += cost
    execute_schedule.last_legs = leg_details

    return complete_path, total_cost, True

## 5. Visualization, Testing & Comparative Analysis
Runs all required test scenarios and compares **BFS** vs. **A\*** across total path cost and total nodes expanded:
1. **Test Run 1:** Custom Environment (`Environment.txt` & `Schedule.txt` — $10\times10$ grid, 12 obstacles, 5 dirty locations, 4 precedence constraints, 3 latest-slot constraints).
2. **Test Run 2:** Provided Sample Environment (`Environment_Sample.txt` & `Schedule_Sample.txt` — $8\times8$ grid, 6 obstacles, 4 dirty locations).
3. **Test Run 3A:** CSP Failure Case (`Environment.txt` & `Schedule_Error.txt` — cyclic precedence constraints).
4. **Test Run 3B:** Navigation Failure Case (`Environment_Error.txt` & `Schedule_Sample.txt` — unreachable dirty location `D1=(0,7)` walled off by obstacles).

In [6]:
def print_grid(grid, charging_station, dirty_locations, path=None, title="Grid"):
    print(f"--- {title} ---")
    rows, cols = grid['rows'], grid['cols']
    path_set = set(path) if path else set()
    dirty_set = set(dirty_locations)

    for r in range(rows):
        row_str = ""
        for c in range(cols):
            pos = (r, c)
            if pos == charging_station:
                row_str += " C "
            elif pos in grid['obstacles']:
                row_str += "###"
            elif pos in dirty_set:
                idx = dirty_locations.index(pos) + 1
                row_str += f" D{idx}"
            elif pos in path_set:
                row_str += " * "
            else:
                row_str += " . "
        print(row_str)
    print("-" * (cols * 3))


def verify_schedule(assignment, constraints):
    # Verify slot exclusivity
    if len(set(assignment.values())) != len(assignment):
        return False
    # Verify precedence
    for d_before, d_after in constraints['precedence']:
        if int(assignment[d_before][1:]) >= int(assignment[d_after][1:]):
            return False
    # Verify latest slot
    for d_task, t_slot in constraints['latest_slots']:
        if int(assignment[d_task][1:]) > int(t_slot[1:]):
            return False
    return True


def run_test_case(env_file, sch_file, label):
    print("=" * 72)
    print(f"TEST CASE: {label}")
    print(f"Files: {env_file} | {sch_file}")
    print("=" * 72)

    try:
        grid, charging_station, dirty_locations = parse_environment(env_file)
        num_dirty = len(dirty_locations)
        constraints = parse_schedule(sch_file, num_dirty)
    except Exception as e:
        print(f"Initialization Error: {e}\n")
        return

    # 1. CSP Scheduling
    variables = [f"D{i+1}" for i in range(num_dirty)]
    domains = {v: [f"T{j+1}" for j in range(constraints['num_slots'])] for v in variables}

    print(f"Grid Size: {grid['rows']}x{grid['cols']} | Charging Station: {charging_station} | Dirty Spots: {num_dirty}")
    print(f"CSP Variables: {variables}")
    print(f"Precedence Constraints: {constraints['precedence']}")
    print(f"Latest-Slot Constraints: {constraints['latest_slots']}")

    assignment = backtracking_search(variables, domains, constraints)

    if assignment is None:
        print("STATUS: [CSP FAILURE] No feasible schedule exists satisfying all constraints.\n")
        return

    sorted_assignment = sorted(assignment.items(), key=lambda x: int(x[1][1:]))
    is_valid = verify_schedule(assignment, constraints)
    print(f"Generated Schedule: {' -> '.join([f'{v}({t})' for v, t in sorted_assignment])} | Constraints Verified: {is_valid}")

    task_map = {f"D{i+1}": dirty_locations[i] for i in range(num_dirty)}
    schedule_coords = [(v, task_map[v]) for v, t in sorted_assignment]
    print(f"Cleaning Visit Order: {[f'{v}:{coord}' for v, coord in schedule_coords]}")

    # 2. Integrated Navigation (BFS vs A*)
    results = {}
    for algo in ["BFS", "A*"]:
        print(f"\n--- Executing Integrated Simulation with {algo} ---")
        complete_path, total_cost, success = execute_schedule(grid, charging_station, schedule_coords, algo)
        legs = getattr(execute_schedule, 'last_legs', [])
        total_nodes = sum(l['nodes'] for l in legs)

        if not success:
            print(f"STATUS: [NAVIGATION FAILURE] Unreachable target encountered using {algo}.")
            results[algo] = {'cost': 'N/A', 'nodes': total_nodes, 'success': False}
            continue

        print(f"STATUS: [SUCCESS] All locations cleaned & returned to Charging Station.")
        print(f"Total Movements / Navigation Cost: {total_cost} | Total Nodes Expanded: {total_nodes}")
        for i, leg in enumerate(legs):
            print(f"  Leg {i+1}: {leg['from']} -> {leg['to']} | Cost: {leg['cost']} | Nodes Expanded: {leg['nodes']}")
            print(f"         Path: {leg['path']}")

        results[algo] = {'cost': total_cost, 'nodes': total_nodes, 'success': True, 'path': complete_path}

    # 3. Grid Visualization & Comparative Table
    if results.get("A*", {}).get('success'):
        print()
        print_grid(grid, charging_station, dirty_locations, title="Initial Environment Grid")
        print_grid(grid, charging_station, [], path=results["A*"]['path'], title="Final Cleaned Grid & Complete Journey (A*)")

    print("\nComparative Analysis (BFS vs. A*):")
    print(f"{'Metric':<22} | {'BFS':<15} | {'A*':<15}")
    print("-" * 58)
    print(f"{'Execution Status':<22} | {str(results['BFS']['success']):<15} | {str(results['A*']['success']):<15}")
    print(f"{'Total Navigation Cost':<22} | {str(results['BFS']['cost']):<15} | {str(results['A*']['cost']):<15}")
    print(f"{'Total Nodes Expanded':<22} | {str(results['BFS']['nodes']):<15} | {str(results['A*']['nodes']):<15}")
    print(f"{'Optimal Path Found':<22} | {'Yes' if results['BFS']['success'] else 'No':<15} | {'Yes' if results['A*']['success'] else 'No':<15}")
    print("\n")


# Execute all 3 required test scenarios
run_test_case("Environment.txt", "Schedule.txt", "1. Custom Environment & Schedule")
run_test_case("Environment_Sample.txt", "Schedule_Sample.txt", "2. Sample Environment & Schedule")
run_test_case("Environment.txt", "Schedule_Error.txt", "3A. Failure Case — Infeasible CSP Schedule")
run_test_case("Environment_Error.txt", "Schedule_Sample.txt", "3B. Failure Case — Unreachable Dirty Location")

TEST CASE: 1. Custom Environment & Schedule
Files: Environment.txt | Schedule.txt
Grid Size: 10x10 | Charging Station: (0, 0) | Dirty Spots: 5
CSP Variables: ['D1', 'D2', 'D3', 'D4', 'D5']
Precedence Constraints: [('D1', 'D3'), ('D2', 'D4'), ('D1', 'D5'), ('D5', 'D4')]
Latest-Slot Constraints: [('D2', 'T3'), ('D1', 'T2'), ('D3', 'T4')]
Generated Schedule: D1(T1) -> D2(T2) -> D3(T3) -> D5(T4) -> D4(T5) | Constraints Verified: True
Cleaning Visit Order: ['D1:(0, 9)', 'D2:(3, 8)', 'D3:(8, 8)', 'D5:(5, 2)', 'D4:(9, 1)']

--- Executing Integrated Simulation with BFS ---
STATUS: [SUCCESS] All locations cleaned & returned to Charging Station.
Total Movements / Navigation Cost: 42 | Total Nodes Expanded: 190
  Leg 1: Charging Station -> D1 (0, 9) | Cost: 9 | Nodes Expanded: 40
         Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
  Leg 2: D1 (0, 9) -> D2 (3, 8) | Cost: 4 | Nodes Expanded: 7
         Path: [(0, 9), (1, 9), (2, 9), (3, 9), (3, 8)]
  Leg 

## 6. Algorithm Design & Theoretical Analysis

### 1. Heuristic Justification (Manhattan Distance)
For 4-directional grid navigation (`UP`, `DOWN`, `LEFT`, `RIGHT`) with unit step cost $c = 1$:
$$h(n) = |r_n - r_{\text{goal}}| + |c_n - c_{\text{goal}}|$$
* **Admissibility:** Without obstacles, the minimum number of orthogonal steps between $(r_n, c_n)$ and $(r_{\text{goal}}, c_{\text{goal}})$ is exactly the Manhattan distance. Obstacles can only force detours that increase the true cost $h^*(n)$, so $h(n) \le h^*(n)$ always holds.
* **Consistency:** Moving one cell changes $|r_n - r_{\text{goal}}|$ or $|c_n - c_{\text{goal}}|$ by at most $1$, so $h(n) \le 1 + h(n')$, guaranteeing optimal paths without re-expanding closed nodes.

### 2. Graph Search Comparison (BFS vs. A\*)
* **BFS** explores frontiers uniformly in concentric layers ($O(b^d)$), guaranteeing shortest paths on unweighted grids but expanding many unnecessary nodes in directions away from the goal.
* **A\*** evaluates $f(n) = g(n) + h(n)$, prioritizing states closer to the target. Both algorithms find identical optimal path costs, while **A\*** expands substantially fewer nodes.

### 3. CSP Formulation & Heuristics
* **MRV + Degree Heuristic:** Selects the most constrained variable first (fewest legal time slots) and breaks ties using the variable involved in the highest number of active constraints with unassigned tasks, triggering fail-fast pruning.
* **LCV (Least Constraining Value):** Orders candidate slots to eliminate the fewest options from neighboring unassigned tasks.
* **Forward Checking:** Immediately prunes inconsistent time slots from unassigned neighbors after each assignment and restores domains cleanly upon backtracking.